<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/sean-1114/notebooks/gsm8k/Experiment_2_Eval_IG_KL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# IG–KL MEDAL vs. AB-MEDAL — Evaluation on 70 GSM8K Questions

This is a copy of the 70-question E2 evaluation workflow with the **same one-step IG–KL reward used in sean-1105** and cooperative question batching matching the original E2 notebook. The original FIG notebook remains available separately.

For a branch `g -> s -> s_next` and the outgoing action that commits position `i`:
$$
R(s,s_{\mathrm{next}}) =
IG(s,s_{\mathrm{next}}) - \lambda\,KL(p_g[i]\,\Vert\,p_s[i]),
\qquad
IG(s,s_{\mathrm{next}})=
\begin{cases}
\dfrac{H(s)-H(s_{\mathrm{next}})}{H(s)}, & H(s)>\epsilon,\\
0, & \text{otherwise}.
\end{cases}
$$

Both `g` and `s` still have a mask at position `i`; KL is measured **before** the outgoing token is committed. Root actions have zero KL penalty. History follows each branch, never the last visited sibling.

Reward defaults match sean-1105: `KL_WEIGHT=0.05`, `KL_TOP_K=32` (previous top-32 token IDs plus one residual-tail bin). **`BATCH_SIZE=2` matches Experiment_2_Eval.ipynb.** Each question keeps its own tree and branch-local KL history; active questions share model forwards. Set top-k to `0` for full-vocabulary KL. The penalty reuses existing forward distributions and adds **zero model forwards**, although KL computation and CPU transfers add overhead. Both UCB backups and AB-MEDAL posterior updates use the combined reward.

The search uses ordinary one-step MEDAL/AB-MEDAL with budgets **256 / 512**, depth **10**, minimum candidates **3**, `K1=3`, `K2=5`, followed by standard deterministic LLaDA confidence completion (256 tokens, T=0). It does not use FIG's virtual-grandchild lookahead or simulation cap. Setting lambda to zero recovers original IG search.

The reward/search protocol matches sean-1105; the evaluation pool expands as follows:
1. **Canonical 20**: the exact original evaluation questions, `sample_slice=(0, 20)`.
2. **Additional 50**: the frozen extended questions, `sample_slice=(20, 70)`.
3. **Combined 70**: paired comparisons and offline E3 figures over both parts.

Results are isolated from FIG and from the old 20-question run, with reward settings recorded in checkpoints and traces. Keep these reward settings fixed on evaluation data.

**Comparison with older runs:** this main-based copy retains the current model loader and answer parser. Main disables BF16 reduced-precision reduction and accepts units after a marked final numeric answer (for example, `#### 110 miles`). These updates and GPU batch numerics can affect outputs or scoring relative to old serial runs. Re-run the canonical 20 here when comparing with the extended 50. `BATCH_SIZE=1` remains available for serial comparison; each batch size gets its own output directory.

## 1. Verify GPU Hardware
Verify CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Total VRAM: {vram_gb:.2f} GiB')


## 2. Repository Setup
Use branch `sean-1114`. Outside Colab, open the notebook in an existing checkout or set `REPO_DIR`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'sean-1114'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')

if not IN_COLAB:
    candidates = ([Path(os.environ['REPO_DIR']).expanduser()] if os.environ.get('REPO_DIR') else []) + [Path.cwd(), *Path.cwd().parents]
    REPO_DIR = next((p.resolve() for p in candidates if (p / 'src/dlm_project').is_dir()), None)
    if REPO_DIR is None:
        raise RuntimeError('Open the notebook inside the sean-1114 checkout or set REPO_DIR.')
    os.chdir(REPO_DIR)
for entry in (str(REPO_DIR), str(REPO_DIR / 'src')):
    if entry not in sys.path:
        sys.path.insert(0, entry)
print('IG-KL checkout:', REPO_DIR)


## 3. Install Dependencies
Install pinned dependencies and editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/dev.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed in Colab.' if IN_COLAB else 'Using the existing local Python environment.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Verify the original IG–KL reward, branch-local history, zero additional model forwards, serial/batch parity on synthetic models, checkpoint equivalence and the frozen 20 + 50 sample split before loading model weights. Real-model GPU batch numerics still need to be measured on the target device.

In [ ]:
import subprocess
import sys

test_env = {**os.environ, 'PYTHONPATH': f"{REPO_DIR / 'src'}:{REPO_DIR}"}
res = subprocess.run(
    [
        sys.executable, '-m', 'pytest',
        'tests/decoding/test_ig_kl_search.py',
        'tests/decoding/test_ig_kl_batch.py',
        'tests/tasks/test_experiment_1_ig_kl.py',
        'tests/tasks/test_ig_kl_evaluation.py',
        '-v',
    ],
    cwd=REPO_DIR, env=test_env, capture_output=True, text=True,
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('IG-KL pre-flight unit tests failed.')
print('IG-KL reward and 70-question sample checks passed.')

## 5. Mount Google Drive & Configure Cache
Use a separate output root for this IG–KL experiment. The next cell refines it by lambda and KL top-k, keeping the original FIG results separate.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_BASE = REPO_DIR / 'results/e2_eval_ig_kl'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_BASE = DRIVE_ROOT / 'results/e2_eval_ig_kl'
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

OUTPUT_BASE = OUTPUT_BASE.resolve()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print(f'IG-KL output root: {OUTPUT_BASE}')

## 6. Experiment Controls & Staged Model Loading
Use one-step IG–KL, lambda 0.05, KL top-32, budgets 256/512, depth 10 and minimum candidates 3. **Batch size 2 matches the original E2 evaluation notebook**; use 1 for serial execution or increase it after checking GPU memory. Load the model once for both evaluation parts. Keep the batch size fixed when resuming a run.

In [ ]:
import shutil
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.ig_kl_evaluation import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context, load_model
)

STAGE = 'evaluation'
METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
REWARD_MODE = 'ig_kl'
KL_WEIGHT = 0.05
KL_TOP_K = 32  # Previous top-k + tail; 0 = full-vocabulary KL.
BATCH_SIZE = 2  # Match Experiment_2_Eval.ipynb; each question has independent KL history.
RESUME = True

OUTPUT_DIR = OUTPUT_BASE / f'lambda_{KL_WEIGHT:g}_topk_{KL_TOP_K}' / f'batch_{BATCH_SIZE}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Reward-specific output:', OUTPUT_DIR)

base_ctx = context(
    stage=STAGE, methods=METHODS, budgets=BUDGETS,
    reward_mode=REWARD_MODE, kl_weight=KL_WEIGHT, kl_top_k=KL_TOP_K,
)
print('Reward settings:', base_ctx['reward'])
print('Frozen evaluation pool:', len(base_ctx['examples']), 'questions')

profile = get_profile('llada')
model_name = profile['repo_id'].split('/')[-1]
drive_model_dir = DRIVE_ROOT / 'models' / model_name
runtime_model_dir = Path('/content/models') / model_name

if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / 'config.json').is_file():
    print(f'Copying model from Google Drive to local runtime storage: {runtime_model_dir}...')
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = os.environ.get('MODEL_PATH') or str(
    runtime_model_dir if runtime_model_dir.is_dir()
    else (drive_model_dir if drive_model_dir.is_dir() else profile['repo_id'])
)
print('Using model location:', MODEL_PATH)
wrapper = load_model(base_ctx, model_path=MODEL_PATH, device='cuda', local_files_only=Path(MODEL_PATH).is_dir())
print('Model identity:', wrapper.identity)

## 7. Execute Part 1: Canonical Evaluation Set (Original 20 Questions)
Run the exact original 20 questions with the sean-1105 IG–KL method. Completed compatible results can be reused with `RESUME=True`; a partially executed search is recomputed.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.ig_kl_evaluation import run_e1_stage

ctx_20 = context(
    stage=STAGE, methods=METHODS, budgets=BUDGETS, sample_slice=(0, 20),
    reward_mode=REWARD_MODE, kl_weight=KL_WEIGHT, kl_top_k=KL_TOP_K,
)
print('Starting Part 1 (Canonical 20 Questions, IG-KL)...')
t0 = perf_counter()
q_rows_20, agg_rows_20 = run_e1_stage(
    wrapper, ctx_20, OUTPUT_DIR / 'canonical_20',
    methods=METHODS, budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH, num_candidates=MIN_CANDIDATES,
    resume=RESUME, batch_size=BATCH_SIZE,
)
print(f'Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')

## 8. Execute Part 2: Extended Validation Set (Additional 50 Questions)
Run the next 50 frozen evaluation questions with identical IG–KL settings, `sample_slice=(20, 70)`.

In [ ]:
ctx_50 = context(
    stage=STAGE, methods=METHODS, budgets=BUDGETS, sample_slice=(20, 70),
    reward_mode=REWARD_MODE, kl_weight=KL_WEIGHT, kl_top_k=KL_TOP_K,
)
print('Starting Part 2 (Additional 50 Questions, IG-KL)...')
t0 = perf_counter()
q_rows_50, agg_rows_50 = run_e1_stage(
    wrapper, ctx_50, OUTPUT_DIR / 'extended_50',
    methods=METHODS, budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH, num_candidates=MIN_CANDIDATES,
    resume=RESUME, batch_size=BATCH_SIZE,
)
print(f'Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')

## 9. Results & Paired Comparisons (Canonical 20, Extended 50, and Combined 70)

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e1_test import compute_paired_comparison
from dlm_project.experiments.sweep_report import write_csv

summaries_dir = OUTPUT_DIR / 'summaries'
summaries_dir.mkdir(parents=True, exist_ok=True)

print('=== Part 1: Canonical 20 Questions Paired Comparison ===')
paired_20 = compute_paired_comparison(q_rows_20, budgets=BUDGETS)
if paired_20:
    df_p20 = pd.DataFrame(paired_20)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_p20)
    write_csv(summaries_dir / 'e2_ig_kl_canonical_20_paired_comparison.csv', paired_20)

print('=== Part 2: Additional 50 Questions Paired Comparison ===')
paired_50 = compute_paired_comparison(q_rows_50, budgets=BUDGETS)
if paired_50:
    df_p50 = pd.DataFrame(paired_50)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_p50)
    write_csv(summaries_dir / 'e2_ig_kl_extended_50_paired_comparison.csv', paired_50)

print('=== Combined Pool (All 70 Questions) Paired Comparison ===')
combined_q_rows = q_rows_20 + q_rows_50
expected_ids = [ex['example_id'] for ex in ctx_20['examples'] + ctx_50['examples']]
expected_keys = {(ex_id, method, budget) for ex_id in expected_ids for method in METHODS for budget in BUDGETS}
actual_keys = [(row['example_id'], row['method'], row['search_budget']) for row in combined_q_rows]
assert len(expected_ids) == len(set(expected_ids)) == 70
assert len(actual_keys) == len(expected_keys) and set(actual_keys) == expected_keys, (
    'Combined-70 reporting requires all 70 questions at both methods and budgets.'
)
paired_combined = compute_paired_comparison(combined_q_rows, budgets=BUDGETS)
if paired_combined:
    df_pcomb = pd.DataFrame(paired_combined)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']]
    display(df_pcomb)
    write_csv(summaries_dir / 'e2_ig_kl_combined_70_paired_comparison.csv', paired_combined)
    write_csv(summaries_dir / 'e2_ig_kl_evaluation_paired_comparison.csv', paired_combined)

assert len(ctx_20['examples']) == 20 and len(ctx_50['examples']) == 50
assert not ({ex['example_id'] for ex in ctx_20['examples']} & {ex['example_id'] for ex in ctx_50['examples']})
write_csv(summaries_dir / 'ig_kl_combined_70_per_question.csv', combined_q_rows)
reward_columns = ['example_id', 'method', 'search_budget', 'selected_ig_reward',
                  'selected_action_history_kl', 'selected_kl_penalty', 'kl_extra_model_calls']
display(pd.DataFrame(combined_q_rows)[reward_columns].head(12))


## 10. Interactive Search Tree & Decision Trace Inspector
Inspect the search tree topology, candidate nodes, and decision trace for any evaluated question.

In [ ]:
from dlm_project.tasks.gsm8k.tree_viz import print_ascii_tree, summarize_decision_trace
import json

# Select a sample question to inspect from canonical_20
sample_ex_id = ctx_20['examples'][0]['example_id']
chk_dir = OUTPUT_DIR / 'canonical_20' / 'evaluation' / 'ab_medal' / sample_ex_id / 'checkpoints/256'
tree_json = chk_dir / 'tree.json'
metrics_json = chk_dir / 'metrics.json'

if tree_json.is_file():
    print(f'=== AB-MEDAL Search Tree at B=256 for {sample_ex_id} ===')
    print(print_ascii_tree(tree_json, max_depth=5, max_children_per_node=3))

if metrics_json.is_file():
    met = json.loads(metrics_json.read_text())
    print(f'=== Checkpoint Metrics ({sample_ex_id} @ B=256) ===')
    print(f'  Selected Node: {met["selected_node_id"]} | Depth: {met["selected_depth"]} | Q: {met["selected_q"]:.4f}')
    print(f'  Exact Match: {met["exact_match"]} | Pred: {met["predicted_answer"]} | Ref: {met["reference_answer"]}')

    print('  Raw IG:', met.get('selected_ig_reward'))
    print('  Historical KL:', met.get('selected_action_history_kl'))
    print('  KL penalty:', met.get('selected_kl_penalty'))
    print('  Extra KL model calls:', met.get('kl_extra_model_calls'))


## 11. E3 Offline Metric Extraction & Publication Figures
Analyze the saved 70-question search trees with zero additional model forwards. Q values and backups use the combined IG–KL reward; **negative IG rates and IG summaries use raw entropy gain** so they are not confused with KL penalties.

In [ ]:
import shutil
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

# Link only complete results for the exact frozen 20+50 questions.
combined_results_dir = OUTPUT_DIR / 'combined_70' / 'evaluation'
combined_results_dir.mkdir(parents=True, exist_ok=True)
for part, part_ctx in [('canonical_20', ctx_20), ('extended_50', ctx_50)]:
    for method in METHODS:
        destination = combined_results_dir / method
        destination.mkdir(parents=True, exist_ok=True)
        for example in part_ctx['examples']:
            q_dir = OUTPUT_DIR / part / 'evaluation' / method / example['example_id']
            for budget in BUDGETS:
                checkpoint = q_dir / 'checkpoints' / str(budget)
                for filename in ('metrics.json', 'tree.json', 'selection.json', 'completion.json'):
                    if not (checkpoint / filename).is_file():
                        raise RuntimeError(f'Finish both evaluation parts before combined E3: missing {checkpoint / filename}')
            target = destination / q_dir.name
            if not target.exists():
                if target.is_symlink():  # Repair a broken relative link from an earlier attempt.
                    target.unlink()
                try:
                    target.symlink_to(q_dir.resolve(), target_is_directory=True)
                except OSError:
                    shutil.copytree(q_dir, target)

eval_results_dir = combined_results_dir
e3_output_dir = OUTPUT_DIR / 'combined_70' / 'summaries'
e3_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Running E3 Offline Tree Analysis on {eval_results_dir}...')
e3_q, e3_agg = analyze_e3_stage(
    results_dir=eval_results_dir,
    output_dir=e3_output_dir,
    stage='evaluation',
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print(f'✓ E3 Analysis Completed successfully: {len(e3_q)} question outcomes evaluated.')

figures_dir = e3_output_dir / 'e3_figures'
fig_titles = {
    'fig1_nfe_by_depth': 'Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets',
    'fig2_branching_factors': 'Figure 2: Branching Factor Distribution — Children per expanded parent',
    'fig3_candidate_density': 'Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth',
    'fig4_root_subtrees': 'Figure 4: Root-Subtree Compute Concentration — Search effort across root branches',
    'fig5_reward_redundancy': 'Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states',
    'fig6_accuracy_vs_budget': 'Figure 6: Accuracy vs Search Budget — Performance scaling curve',
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f'{fig_key}.png'
    if png_path.is_file():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(png_path), width=750))


## 12. Synchronize All Artifacts to Google Drive
Flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
